**自动微分**<a id='toc0_'></a>    
- [一个简单的例子](#toc1_)    
- [非标量变量的反向传播：单独计算批量中每个样本的偏导数之和](#toc2_)    
- [分离计算](#toc3_)    
- [Python控制流的梯度计算](#toc4_)    
- [小结](#toc5_)    
- [练习](#toc6_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<div style="font-size:32px;  padding:5px;"> 
自动微分
</div>


正如[微积分](calculus.ipynb)中所说，求导是几乎所有深度学习优化算法的关键步骤。
虽然求导的计算很简单，只需要一些基本的微积分。
但对于复杂的模型，手工进行更新是一件很痛苦的事情（而且经常容易出错）。

深度学习框架通过自动计算导数，即*自动微分*（automatic differentiation）来加快求导。
实际中，根据设计好的模型，系统会构建一个*计算图*（computational graph），
来跟踪计算是哪些数据通过哪些操作组合起来产生输出。

为了计算导数，自动微分通过反向遍历这个计算图并应用链式法则来实现。这种应用链式法则的计算算法被称为**反向传播**（backpropagate），其核心是：通过遍历整个计算图，逐一计算并记录每个模型参数对应的偏导数，为后续的参数更新提供依据。

虽然自动微分库在过去十年中成为热门关注点，但它们其实有着悠久的历史。事实上，关于自动微分的最早参考文献可以追溯到半个多世纪以前 :cite:`Wengert.1964`。现代反向传播背后的核心思想源于1980年的一篇博士论文 :cite:`Speelpenning.1980`，并在20世纪80年代末得到了进一步发展 :cite:`Griewank.1989`。虽然反向传播已成为计算梯度的默认方法，但它并非唯一选择。例如，Julia编程语言就采用了前向传播 :cite:`Revels.Lubin.Papamarkou.2016`。在探索各种方法之前，让我们首先掌握autograd包的使用。



❓【补充】什么是反向传播

考虑最简单的三层复合函数：

$$
x \xrightarrow{\;u\;} 
z = w x
\xrightarrow{\;v\;}
\hat y = \sigma(z)
\xrightarrow{\;L\;}
L = \ell(\hat y)
$$

目标：求  
$$
\frac{dL}{dw}
$$

💡 参考数学分析知识，有
$$
\frac{dL}{dw}
=
\frac{dL}{d\hat y}
\cdot
\frac{d\hat y}{dz}
\cdot
\frac{dz}{dw}
$$

这正是 **链式法则**，也是反向传播的核心计算方式。

- **前向传播计算函数值**：
$$
  x \to z \to \hat y \to L
$$

- **反向传播计算梯度**：
$$
  \frac{dL}{d\hat y}
  \to
  \frac{dL}{dz}
  \to
  \frac{dL}{dw}
$$

这种求解梯度（而非计算函数值）的算法，正是 “backpropagation” 中 **back** 的含义。

🔑 反向传播实现步骤：
1. 每一层只需要计算当前层的局部导数（比如激活函数的导数、线性变换的偏导）
2. 和后一层传来的梯度相乘（链式法则的核心）
3. 最终得到当前层所有参数的偏导数，其时间复杂度和网络层数成正比（O(n)）。而手动计算是指数级复杂度

📖 【通俗类比】不妨把神经网络想象成一家工厂：
* 正向传播：原材料（输入$x$）经过多道工序（各层变换）变成产品（输出$\hat{y}$）
* 损失函数：产品的质量缺陷（和真实值$y$的差距）
* 反向传播：从最终的质量缺陷出发，反向检查每道工序的问题：
    * 先看最后一道工序（输出层）：是不是机器参数$(w_2,b_2)$没调好？
    * 再看前一道工序（隐藏层）：是不是上一道的半成品$a_1$质量有问题？还是机器参数$(w_1,b_1)$的问题？
    * 这样量化分析参数的影响，最终针对性调整所有机器参数。



# <a id='toc1_'></a>[一个简单的例子](#toc0_)

作为一个演示例子，(**假设我们想对函数$y=2\mathbf{x}^{\top}\mathbf{x}$关于列向量$\mathbf{x}$求导**)。
首先，我们创建变量`x`并为其分配一个初始值。

In [1]:
import torch

x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

在我们计算$y$关于$\mathbf{x}$的梯度之前，需要一个地方来存储梯度。

💡 重要的是，我们不会在每次对一个参数求导时都分配新的内存。

$\implies$ 我们经常会成千上万次地更新相同的参数，每次都分配新的内存可能很快就会将内存耗尽。

💡 注意，一个【标量函数】关于【向量】$\mathbf{x}$的【梯度是向量】，并且与$\mathbf{x}$具有相同的形状。


In [2]:
# 方法名后面带下划线 _，代表原地操作张量x
x.requires_grad_(True)  #直接修改 x 这个对象内部的属性。把x作为变量建立一张计算图（Computational Graph），（如果所有的张量这个属性都是false那么就不建立计算图）等价于x=torch.arange(4.0,requires_grad=True)
x.grad  # 默认值是None，用于储存梯度

(**现在计算$y$。**)


In [3]:
y = 2 * torch.dot(x, x)# 设置y
y

tensor(28., grad_fn=<MulBackward0>)

`x`是一个长度为4的向量，计算`x`和`x`的点积，得到了我们赋值给`y`的标量输出。

接下来[**我们现在可以通过调用`y`的`backward`方法来计算`y`关于`x`的梯度**]。

并且通过`x`的`grad`属性来访问这个梯度。


In [4]:
y.backward()#反向传播计算梯度
# 拿着记录表，沿着运算路径倒着走回去。
# 根据微积分中的链式法则，pytorch能精确算出 x 的微小变化会对 y 产生多大影响。这个影响的值，就是梯度，存在 x.grad 中
x.grad

tensor([ 0.,  4.,  8., 12.])

函数$y=2\mathbf{x}^{\top}\mathbf{x}$关于$\mathbf{x}$的梯度应为$4\mathbf{x}$。
让我们快速验证这个梯度是否计算正确。


In [5]:
x.grad == 4 * x# 进行验证

tensor([True, True, True, True])

【**现在让我们计算`x`的另一个函数并求其梯度。**】

需要注意的是，当我们记录新梯度时，PyTorch不会自动重置梯度缓冲区。而是将新的梯度累加到已存储的梯度上。
当我们想要优化多个目标函数的和时，这种行为会很有用。

要重置梯度缓冲区，我们可以按如下方式调用`x.grad.zero_()`：

In [6]:
# 在默认情况下，PyTorch会累积梯度，我们需要清除之前的值
x.grad.zero_()
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

In [7]:
# 重新设定x为新张量  
x = torch.tensor([5.0], requires_grad=True)  
print(x.grad)  # None → 新张量无初始梯度  
y_new = x*2  
y_new.backward()  
print(x.grad)  # tensor([2.0]) → 新梯度正常生成

None
tensor([2.])


# <a id='toc2_'></a>[非标量变量的反向传播：单独计算批量中每个样本的偏导数之和](#toc0_)

当`y`是一个向量时，`y`关于向量`x`的导数最直观的表示形式是一个被称为**雅可比矩阵**（Jacobian）的矩阵，其中包含了`y`的每个分量关于`x`的每个分量的偏导数。同样地，对于更高维度的`y`和`x`，求导的结果可能是一个更高阶的张量（tensor）。

虽然雅可比矩阵会出现在一些高级机器学习技术中，但更常见的操作是对`y`的每个分量关于完整向量`x`的梯度进行求和，得到一个与`x`形状相同的向量。

🌰 我们通常会有一个向量，代表在一批（batch）训练样本中为每个样本单独计算得到的损失函数值。

🌸 在这种情况下，我们只需要（**将为每个样本单独计算出的梯度进行求和**）。


⚠️使用pytorch的说明：

由于深度学习框架对非标量张量梯度的解释方式各不相同，PyTorch采取了一些措施来避免混淆。

在非标量上调用`backward`方法会引发错误，除非我们告诉PyTorch如何将其简化为标量。更正式地说，我们需要提供一个向量$\mathbf{v}$，这样`backward`方法将计算$\mathbf{v}^\top \partial_{\mathbf{x}} \mathbf{y}$，而不是$\partial_{\mathbf{x}} \mathbf{y}$。

这部分内容可能有些令人困惑，但出于后续会明晰的原因，这个代表$\mathbf{v}$的参数被命名为`gradient`。如需更详细的说明，请参考杨张在Medium上的文章：[《PyTorch backward函数中的gradient参数实例解析》](https://zhang-yang.medium.com/the-gradient-argument-in-pytorchs-backward-function-explained-by-examples-68f266950c29)。

In [7]:
# 对非标量调用backward需要传入一个gradient参数，该参数指定微分函数关于self的梯度。
x.grad.zero_()
y = x * x # 因为y是非标量，直接调用y.backward()会报错，必须指定gradient参数
y.backward(gradient=torch.ones(len(y)))  # 等价于y.sum().backward()，但后者更好而且往往更快
# 因为前者新创建了一个与y形状相同的张量gradient，其每个元素用以对应y中元素的权重系数。在张量大的时候会非常吃显存
# 并且torch.ones() 默认在 CPU 上创建张量而不是在gpu上操作，保险起见最好写作
# y.backward(gradient=torch.ones_like(y)) 才能保证设备一致
# 后者的sum() 操作将张量规约为一个标量。这个操作非常快且几乎不占内存。且会自动保留在原设备上。
# 在反向传播（Backward pass）时，PyTorch 内部知道 Sum 操作的导数就是全是 1 的梯度。
# PyTorch 的底层实现通常会通过广播（broadcasting）或优化内核直接处理这种情况，而不需要显式地分配一个巨大的全是 1 的张量。
# 这一步实际计算的是：sum(gradient[i] * y[i]) 对x的梯度（即Jacobian向量积）
x.grad

tensor([0., 2., 4., 6.])

对比`y.backward(gradient=torch.ones(len(y))) `和`y.sum().backward()`两种书写方式，现在进行数学等价性验证：

根据链式法则（Chain Rule）：
$$
\frac{\partial \text{loss}}{\partial x} = \frac{\partial \text{loss}}{\partial y} \cdot \frac{\partial y}{\partial x}
$$

1. **使用 `gradient=torch.ones`：**
   显式地定义了 $\frac{\partial \text{loss}}{\partial y} = [1, 1, \dots, 1]$。
   计算结果为：$1 \cdot \frac{\partial y_1}{\partial x} + 1 \cdot \frac{\partial y_2}{\partial x} + \dots$

2. **使用 `y.sum()`：**
   令虚拟的 Loss $L = \sum y_i$。
   那么 $\frac{\partial L}{\partial y_i} = 1$。
   计算结果依然是：$\sum \frac{\partial y_i}{\partial x}$。

$\implies$ 两者在数学上具有等效性。

# <a id='toc3_'></a>[分离计算](#toc0_)

有时，我们希望【**将某些计算移动到记录的计算图之外**】。
例如，假设`y`是作为`x`的函数计算的，而`z`则是作为`y`和`x`的函数计算的。
想象一下，我们想计算`z`关于`x`的梯度，但由于某种原因，希望将`y`视为一个常数，
并且只考虑到`x`在`y`被计算后发挥的作用。

这里可以分离`y`来返回一个新变量`u`，该变量与`y`具有相同的值，
但


有时候，我们希望将某些计算**移到已记录的计算图之外**。

🌰 假设我们基于输入项创建一些中间项用以辅助计算，但不希望计算这些辅助项的梯度。这时就需要合理规划计算图，将辅助项分离

🌰 假设我们有`z = x * y`和`y = x * x`。我们希望得到`x`对`z`的**直接**影响，忽视`x`通过`y`传递的间接影响。因此创建了一个新变量`u`。它的值与`y`相同，但会丢弃计算图中关于`y`的信息。

换句话说，此时是把`u`视为常数（在计算图中没有父节点），阻止梯度通过`u`流向`x`。计算出来的结果就不会是`z=x*x*x`关于`x`的偏导数。

In [8]:
x.grad.zero_()
y = x * x
u = y.detach()# 指名这个为常数
z = u * x

z.sum().backward()# 这计算z = ux
x.grad == u

tensor([True, True, True, True])

需要注意的是，此时指向`y`的计算图仍然保留，因此我们仍然可以计算`y`关于`x`的梯度，即`2*x`。

In [9]:
x.grad.zero_()
y.sum().backward()#这是计算y=xx
x.grad == 2 * x

tensor([True, True, True, True])

# <a id='toc4_'></a>[Python控制流的梯度计算](#toc0_)

到目前为止，我们讨论的都是输入到输出的路径可以通过`z = x * x * x`这类函数明确定义的情况。但编程为我们提供了更多计算结果的灵活方式。例如，我们可以让计算过程依赖辅助变量，或者根据中间结果进行条件选择。

使用自动微分的一个优势在于：**即使**构建函数的计算图需要经过复杂的Python控制流（例如条件判断、循环和任意函数调用），我们**仍然可以计算最终变量的梯度**。

为了说明这一点，考虑下面的代码片段：其中`while`循环的迭代次数和`if`语句的判断逻辑都依赖于输入`a`的值。


In [10]:
def f(a):# 分段线性函数
    b = a * 2
    while b.norm() < 1000:#直到b的绝对值不小于1000，k次后b = 2^(k+1)*a
        b = b * 2
    #此时再进行
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c
# 此时c = D*b，D=1或100
# 也就是c = D*b = D*2^(k+1)*a = K*a

接下来，我们传入一个随机值作为输入来调用这个函数。由于输入是随机变量，我们无法预知计算图的具体结构。然而，每当我们针对某个特定输入执行`f(a)`时，都会生成一个具体的计算图，随后我们就可以运行`backward`来计算梯度。


In [11]:
a = torch.randn(size=(), requires_grad=True)#size=()代表是标量，内容是随机生成的
d = f(a)
d.backward()

尽管我们的函数`f`是为了演示而刻意设计的，但它对输入的依赖关系其实非常简单：它是`a`的**线性**函数，只是缩放系数是分段定义的。因此，`f(a) / a`是一个元素均为常数的向量，而且`f(a) / a`必须与`f(a)`关于`a`的梯度值相等。

In [12]:
a.grad == d / a#找出梯度，此时a.grad = K且d/a = K

tensor(True)

动态控制流在深度学习中极为常见。例如，处理文本数据时，计算图的结构会随输入文本的长度变化而改变。在这类场景下，自动微分对统计建模而言至关重要——因为我们无法预先计算梯度。

# <a id='toc5_'></a>[小结](#toc0_)
至此，你已经初步领略了自动微分的强大之处。自动、高效的导数计算库的发展，极大提升了深度学习从业者的工作效率，让他们得以从繁琐的机械性工作中解脱出来，专注于更具创造性的任务。此外，自动微分工具支持我们设计超大规模模型，这类模型的梯度若用手动推导，耗时将达到难以承受的程度。
 
有趣的是，尽管我们用自动微分来**优化模型**（统计意义上的优化），但自动微分库自身的**优化**（计算层面的优化）也是框架开发者关注的核心课题。开发者会借助编译器和图操作工具，以最快捷、最节省内存的方式计算结果。
 
请先记住以下基础步骤：
(i) 为需要计算偏导数的变量附加梯度；
(ii) 记录目标值的完整计算过程；
(iii) 执行反向传播函数；
(iv) 获取最终计算得到的梯度。

# <a id='toc6_'></a>[练习](#toc0_)

1. 为什么二阶导数的计算成本远高于一阶导数？
2. 运行反向传播函数后，立即再次运行它，观察会发生什么并分析原因。
3. 在之前通过控制流计算`d`关于`a`的导数的示例中，如果将变量`a`改为随机向量或矩阵会发生什么？此时`f(a)`的结果不再是标量，计算结果会有什么变化？我们该如何分析这种情况？
4. 设函数 $f(x) = \sin(x)$，绘制函数 $f(x)$ 及其导数 $f'(x)$ 的图像。要求不利用 $f'(x) = \cos(x)$ 的已知结论，仅通过自动微分得到结果。
5. 设函数 $f(x) = ((\log x^2) \cdot \sin x) + x^{-1}$，绘制从 $x$ 到 $f(x)$ 的依赖关系图。
6. 使用链式法则计算上述函数的导数 $\frac{df}{dx}$，并将每一项的计算结果标注在你之前绘制的依赖关系图上。
7. 基于依赖图和中间导数结果，计算梯度有两种路径：一种从 $x$ 出发正向计算到 $f(x)$，另一种从 $f(x)$ 反向追溯到 $x$。前者通常被称为**前向微分**，后者被称为**反向微分**。分别用两种路径计算结果。
8. 什么时候适合使用前向微分，什么时候适合使用反向微分？提示：考虑所需的中间数据量、步骤的并行化能力，以及涉及的矩阵和向量规模。


1. Why is the second derivative much more expensive to compute than the first derivative?
1. After running the function for backpropagation, immediately run it again and see what happens. Investigate.
1. In the control flow example where we calculate the derivative of `d` with respect to `a`, what would happen if we changed the variable `a` to a random vector or a matrix? At this point, the result of the calculation `f(a)` is no longer a scalar. What happens to the result? How do we analyze this?
1. Let $f(x) = \sin(x)$. Plot the graph of $f$ and of its derivative $f'$. Do not exploit the fact that $f'(x) = \cos(x)$ but rather use automatic differentiation to get the result. 
1. Let $f(x) = ((\log x^2) \cdot \sin x) + x^{-1}$. Write out a dependency graph tracing results from $x$ to $f(x)$. 
1. Use the chain rule to compute the derivative $\frac{df}{dx}$ of the aforementioned function, placing each term on the dependency graph that you constructed previously. 
1. Given the graph and the intermediate derivative results, you have a number of options when computing the gradient. Evaluate the result once starting from $x$ to $f$ and once from $f$ tracing back to $x$. The path from $x$ to $f$ is commonly known as *forward differentiation*, whereas the path from $f$ to $x$ is known as backward differentiation. 
1. When might you want to use forward, and when backward, differentiation? Hint: consider the amount of intermediate data needed, the ability to parallelize steps, and the size of matrices and vectors involved.

[Discussions](https://discuss.d2l.ai/t/1759)
